# Exploratory Data Analysis — Healthcare Master Dataset

**Project:** Healthcare Analytics  
**Author:** Radha Yadav  

---

## Objective
Perform SQL-based exploratory analysis on the healthcare master dataset to extract business insights on patient demographics, revenue, treatment patterns, and appointment behavior.

## Approach
Data is loaded into an in-memory SQLite database to demonstrate SQL querying skills alongside Python analysis.

## Analysis Areas
1. Dataset Overview
2. Patient Demographics
3. Revenue Analysis
4. Treatment Analysis
5. Payment Behavior
6. Monthly Trends
7. Top Patients by Revenue

## 1. Dataset Overview

Load the cleaned master dataset into SQLite for SQL-based analysis.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
master_data = pd.read_csv('../data/cleaned/master_healthcare_dataset.csv')
print('Dataset Shape:', master_data.shape)
print('Columns:', master_data.columns.tolist())
master_data.head()

In [ ]:
# Load into SQLite for SQL analysis
conn = sqlite3.connect(':memory:')
master_data.to_sql('master_healthcare', conn, if_exists='replace', index=False)

# Verify
result = pd.read_sql('SELECT COUNT(*) AS total_records FROM master_healthcare', conn)
print('Total records loaded:', result['total_records'][0])

## 2. Patient Demographics

Understand patient distribution by gender and other demographic factors.

In [ ]:
query = """
SELECT gender,
       COUNT(*) AS total_patients
FROM master_healthcare
GROUP BY gender
"""
gender_dist = pd.read_sql(query, conn)
print('Patient Distribution by Gender:')
print(gender_dist)

plt.figure(figsize=(6, 4))
sns.barplot(x='gender', y='total_patients', data=gender_dist)
plt.title('Patient Count by Gender')
plt.tight_layout()
plt.show()

## 3. Revenue Analysis

Analyze total revenue, average costs, and revenue breakdown by payment method.

In [ ]:
# Total revenue and average cost
query = """
SELECT 
    SUM(amount)  AS total_revenue,
    AVG(cost)    AS average_cost,
    COUNT(*)     AS total_transactions
FROM master_healthcare
"""
print('Overall Revenue Summary:')
pd.read_sql(query, conn)

In [ ]:
# Revenue by payment method
query = """
SELECT payment_method,
       SUM(amount)  AS total_revenue,
       COUNT(*)     AS transactions
FROM master_healthcare
GROUP BY payment_method
ORDER BY total_revenue DESC
"""
payment_rev = pd.read_sql(query, conn)
print('Revenue by Payment Method:')
print(payment_rev)

plt.figure(figsize=(8, 5))
sns.barplot(x='payment_method', y='total_revenue', data=payment_rev)
plt.title('Total Revenue by Payment Method')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Treatment Analysis

Identify which treatment types are most common and most expensive.

In [ ]:
# Most common treatments
query = """
SELECT treatment_type,
       COUNT(*) AS total
FROM master_healthcare
GROUP BY treatment_type
ORDER BY total DESC
"""
treatments = pd.read_sql(query, conn)
print('Treatment Type Distribution:')
print(treatments)

plt.figure(figsize=(10, 5))
sns.barplot(x='treatment_type', y='total', data=treatments)
plt.title('Most Common Treatment Types')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Most expensive treatments
query = """
SELECT treatment_type,
       MAX(cost) AS highest_cost,
       AVG(cost) AS avg_cost
FROM master_healthcare
GROUP BY treatment_type
ORDER BY highest_cost DESC
"""
print('Treatment Cost Analysis:')
pd.read_sql(query, conn)

## 5. Payment Behavior

Analyze payment status distribution — how many patients have paid vs pending.

In [ ]:
query = """
SELECT payment_status,
       COUNT(*) AS total
FROM master_healthcare
GROUP BY payment_status
"""
payment_status = pd.read_sql(query, conn)
print('Payment Status Distribution:')
print(payment_status)

plt.figure(figsize=(6, 4))
plt.pie(payment_status['total'], labels=payment_status['payment_status'], autopct='%1.1f%%', startangle=90)
plt.title('Payment Status Distribution')
plt.tight_layout()
plt.show()

## 6. Monthly Trends

Identify which months see the highest patient volume — useful for resource planning.

In [ ]:
query = """
SELECT month,
       COUNT(*) AS total_treatments
FROM master_healthcare
GROUP BY month
ORDER BY month
"""
monthly = pd.read_sql(query, conn)
print('Monthly Treatment Volume:')
print(monthly)

plt.figure(figsize=(10, 5))
sns.lineplot(x='month', y='total_treatments', data=monthly, marker='o')
plt.title('Monthly Treatment Volume Trend')
plt.xlabel('Month')
plt.ylabel('Total Treatments')
plt.tight_layout()
plt.show()

## 7. Top Patients by Revenue

Identify the top 5 highest-billing patients.

In [ ]:
query = """
SELECT patient_id_x,
       SUM(amount) AS total_bill
FROM master_healthcare
GROUP BY patient_id_x
ORDER BY total_bill DESC
LIMIT 5
"""
top_patients = pd.read_sql(query, conn)
print('Top 5 Highest Billing Patients:')
print(top_patients)

---
## Summary of Key Insights

| Analysis Area | Key Finding |
|---|---|
| Demographics | Gender distribution across patient base |
| Revenue | Total revenue and top payment methods identified |
| Treatments | Most common and highest-cost treatment types |
| Payments | Paid vs pending payment breakdown |
| Monthly Trends | Peak months for patient volume identified |
| Top Patients | Top 5 patients contribute disproportionate revenue |

> **Note:** This notebook uses SQLite for in-memory querying. Advanced SQL analysis (window functions, CTEs, stored procedures) is in `/sql/healthcare_queries.sql` using PostgreSQL.